# Midterm Demo — Building an AI-Powered Operations Dashboard
### BUSAD 878 · Week 6 · Supply-Chain edition

This notebook is a **worked example** of the midterm project. By the end you will have:

1. connected to a dataset (here: simulated supply-chain shipments),
2. validated it so bad rows can't poison your KPIs,
3. built an **insightful dashboard** (charts that tell a story),
4. implemented **three generative-AI features**, and
5. launched the whole thing as an **interactive web app**.

It also sets up the **build-vs-buy** comparison from this week's lesson: here we *build* with
code; the companion `Lovable_NoCode_Dashboard_Guide.md` *buys* the same result by describing it
in plain English.

> **Design idea to watch for:** the AI never does arithmetic. It translates your question into a
> *plan* and narrates *pre-computed* numbers. Pandas does the math. That separation is what makes
> the AI's output checkable — and it's exactly what the rubric rewards.

## 0 · Setup

Run the demo folder's `requirements.txt` first (`pip install -r requirements.txt`). If you launched
Jupyter from a different folder, set `HERE` to this folder so the imports resolve.

In [ ]:
import os, sys, json
HERE = os.getcwd()
# If imports fail, uncomment and point this at the demo folder:
# HERE = r"D:\Penn State Great Valley\BUSAD 878\Week 6\Midterm Project Demo"
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import pandas as pd
import plotly.express as px

import ai
from connectors import load_data, clean_and_validate
from analytics import compute_metric, run_query_plan, detect_anomalies

pd.set_option("display.max_columns", 30)
print("AI mode:", "Gemini (live)" if ai.ai_available() else "offline (rule-based)",
      "| model:", ai.MODEL)

## 1 · Connect to data

`connectors.load_data()` is the single front door. `source="synthetic"` works out of the box;
`csv`, `google_sheet`, `sql`, `rest_api`, and `kaggle` are **placeholders** showing exactly where
you'd plug in your own project's data. **For your midterm, you change one line here** — everything
downstream is identical.

If you simulate data, submit your generator. Ours is `generate_synthetic_data.py`, and its
docstring spells out every assumption (the story, the distributions, and the *deliberately
injected* data-quality problems).

In [ ]:
raw = load_data("synthetic")          # <-- swap to load_data("csv", path="...") for your data
print(f"{len(raw):,} rows, {raw.shape[1]} columns")
raw.head(3)

## 2 · Validate (error handling, made visible)

Real data is dirty. `clean_and_validate()` parses types and **flags** impossible rows — negative
quantities, deliveries before the order date — instead of crashing or silently dropping them. Bad
rows are marked `is_valid=False` and excluded from metrics, with a human-readable report.

This is the "graceful handling of bad inputs" the rubric asks for.

In [ ]:
df, dq_report = clean_and_validate(raw)
print("Validation report:")
print(json.dumps(dq_report, indent=2))

print("\nThe rows we flagged (and why):")
df.loc[~df.is_valid, ["order_id","units_ordered","units_received",
                      "order_date","actual_delivery_date","dq_issue"]]

## 3 · The insightful dashboard (charts that tell a story)

A dashboard isn't a pile of charts — it's an argument. Start with the headline, then let each chart
peel back a layer. Watch what happens: the **headline looks fine**, but the detail says otherwise.

In [ ]:
print("Headline KPIs")
for m, lbl in [("otif_rate","OTIF %"),("on_time_rate","On-time %"),
               ("in_full_rate","In-full %"),("avg_lead_time","Avg lead (days)")]:
    print(f"  {lbl:>16}: {compute_metric(df, m):.1f}")

In [ ]:
d = df[df.is_valid & df.status.eq("Delivered")]

# (a) OTIF by month — looks broadly stable around the low-80s...
trend = d.groupby("month")["otif"].mean().mul(100).round(1).reset_index()
fig = px.line(trend, x="month", y="otif", markers=True, title="OTIF by month (overall)")
fig.add_hline(y=95, line_dash="dot", annotation_text="95% target")
fig.update_yaxes(range=[0,100]); fig.show()

In [ ]:
# (b) ...but break it out by supplier and the spread is huge.
g = d.groupby("supplier")["otif"].mean().mul(100).round(1).sort_values().reset_index()
px.bar(g, x="otif", y="supplier", orientation="h", color="otif",
       color_continuous_scale="RdYlGn", title="OTIF by supplier",
       labels={"otif":"OTIF %","supplier":""}).show()

In [ ]:
# (c) Region x mode: Ocean lanes and APAC lag.
g = (d.groupby(["supplier_region","transport_mode"])["otif"].mean()
       .mul(100).round(1).reset_index())
px.bar(g, x="supplier_region", y="otif", color="transport_mode", barmode="group",
       title="OTIF by region & transport mode", labels={"otif":"OTIF %","supplier_region":""}).show()

In [ ]:
# (d) Cost: a Raw Materials spike appears at the end of the window.
g = df[df.is_valid].groupby(["month","product_category"])["unit_cost"].mean().round(2).reset_index()
px.line(g, x="month", y="unit_cost", color="product_category",
        title="Average unit cost by category", labels={"unit_cost":"Avg unit cost ($)","month":""}).show()

> **The story:** the company-wide OTIF average is *masking* problems. That's the gap the AI features
> below are built to close — and the narrative your reflection document should tell.

## 4 · AI Feature 1 — Natural-language querying

The pattern: **LLM → JSON plan → validate → pandas executes → LLM narrates the real numbers.**
The model proposes *what to compute*; it never computes. A hallucinated column or bad metric is
rejected by `validate_plan()` before it can run. Open the plan to see exactly what it asked for.

In [ ]:
res = ai.nl_query(df, "Which region has the worst OTIF in peak season?")
print(f"[{res['mode']}] {res['answer']}\n")
print("Query plan the model proposed (then we validated + executed it):")
print(json.dumps(res["plan"], indent=2))
res["table"]

In [ ]:
# Try a few more — works the same in offline mode.
for q in ["Average lead time by transport mode",
          "OTIF by supplier for APAC",
          "Total order value by product category"]:
    r = ai.nl_query(df, q)
    print(f"Q: {q}\n   [{r['mode']}] {r['answer']}\n")

## 5 · AI Feature 2 — Automated insights

We hand the model **pre-computed aggregates** (not raw rows) and ask for the few things a manager
should notice. Feeding summaries, not data dumps, keeps it cheap, fast, and grounded.

In [ ]:
out = ai.auto_insights(df, n=5)
print(f"[source: {out['mode']}]")
for b in out["insights"]:
    print(" •", b)

## 6 · AI Feature 3 — Anomaly detection + explanation

Statistics find the anomalies (deterministic, in `analytics.detect_anomalies`); the LLM *explains*
them and suggests actions. Three detectors run: **cost spikes**, **month-on-month OTIF drops**, and
**gradual declining trends** — the last one catches the slow-degrading supplier that a single
month-vs-baseline check would miss.

In [ ]:
ex = ai.explain_anomalies(df)
print(f"[source: {ex['mode']}]")
display(ex["anomalies"])
print(ex["explanation"])

## 7 · Make it interactive (in-notebook widgets)

`ipywidgets` turns a function into live controls — a quick way to explore before committing to the
full web app. Pick a region and the chart redraws; type a question and it's answered live.

In [ ]:
try:
    from ipywidgets import interact, Dropdown, Text
    regions = ["(all)"] + sorted(df.supplier_region.unique())

    @interact(region=Dropdown(options=regions, value="(all)"))
    def show_supplier_otif(region):
        sub = d if region == "(all)" else d[d.supplier_region == region]
        g = sub.groupby("supplier")["otif"].mean().mul(100).round(1).sort_values().reset_index()
        px.bar(g, x="otif", y="supplier", orientation="h", range_x=[0,100],
               title=f"OTIF by supplier — {region}").show()
except ImportError:
    print("ipywidgets not installed — run: pip install ipywidgets")

In [ ]:
try:
    from ipywidgets import Text
    @interact(question=Text(value="on-time rate by supplier in Q4", description="Ask:"))
    def ask(question):
        if question.strip():
            r = ai.nl_query(df, question)
            print(f"[{r['mode']}] {r['answer']}")
            return r["table"]
except ImportError:
    pass

## 8 · Launch it as a web app

The notebook is great for *building*; a **Streamlit app** is how you *ship* an interactive tool a
non-technical colleague can actually use. `app.py` reuses everything above — same connectors, same
analytics, same `ai.py`.

**The normal way** (in a terminal, from this folder):
```bash
streamlit run app.py
```
Then open the URL it prints (default **http://localhost:8501**).

You can also launch it from here in the background:

In [ ]:
# OPTIONAL: launch the Streamlit app in the background from the notebook.
# Stop it later with  proc.terminate()
import subprocess, sys, time
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.headless=true"],
    cwd=HERE)
time.sleep(3)
print("Streamlit launching… open http://localhost:8501")
print("When done:  proc.terminate()")

## 9 · Where this connects to the midterm

- **Build vs. buy.** You just *built* this. The `Lovable_NoCode_Dashboard_Guide.md` *buys* the same
  dashboard by describing it. Both are legitimate — your reflection should argue which fits your
  team, data sensitivity, and timeline (the framework is in the Week 6 lesson).
- **Jagged frontier.** NL querying and narration are squarely inside AI's strengths; trusting it to
  *do the math* would be over the edge — which is why we don't.
- **Augmentation, not automation.** The dashboard surfaces and explains; a human still decides what
  to do about Pacific Components.

**Deliverables to produce next:** the working prototype (this app), a short technical doc (prompts +
design decisions + limits), a ≤5-min demo video, and the **reflection document**
(`Midterm_Reflection_Document_Template.docx`). The grading rubric is in `Grading Rubric.docx`.